In [0]:
df = spark.read.table("workspace.bronze.erp_loc_a101")

In [0]:
df.display()

In [0]:
## trim the whitespaces, change name of the column and remove hyphen from the first column 
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col
from pyspark.sql.types import StringType

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df1 = df.withColumn(field.name, trim(col(field.name)))

mapping = {
    'CID': 'customer_key',
    'CNTRY': 'country'
}
df2 = df1.withColumnsRenamed(mapping)
df3 = df2.withColumn("customer_key", F.regexp_replace(col("customer_key"), "-", ""))
df3.display()

In [0]:
## check for duplicates, if no duplicate values then write to delta table
duplicate_count = df3.groupBy(df3.columns[0]).count().filter("count > 1")
display(duplicate_count)

In [0]:
## check for duplicates, if no duplicate values then write to delta table
duplicate_count = df3.groupBy(df3.columns[0]).count().filter("count > 1")
display(duplicate_count)

## write to silver table 

In [0]:
df3.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.erp_customer_location_cleaned")